# Fig. 2/Fig. 3 Fas3/Vsg StarDist ring-canal segmentation

Cleaned from `RingCanals/Code/StardistPrediction_Fas3_Vsg.ipynb`. This notebook segments Vsg-positive ring canals in Fas3/Vsg image stacks used for pnut(RNAi) and shrb(RNAi) ring-canal quantification.

The source notebook loaded the `ring_canals` StarDist model, now archived at `../Models/Stardist/ring_canals`. The `3D_Stardist_FINETUNED` model is archived separately because it is used by the related shrb Fas3/DAPI segmentation notebook, but it was not the model loaded by this Fas3/Vsg notebook.


## Imports

Figure association: upstream segmentation for Fig. 2L and Fig. 3E measurements.


In [ ]:
from __future__ import annotations

import gc
from pathlib import Path

import matplotlib
matplotlib.rcParams["image.interpolation"] = "None"
import matplotlib.pyplot as plt
import numpy as np
from csbdeep.io import save_tiff_imagej_compatible
from csbdeep.utils import normalize
from skimage import filters
from stardist import random_label_cmap
from stardist.models import StarDist3D
from tifffile import imread

from utils.ImagingFunctions import subtract_background
from utils.SplitLabels import SplitLabels

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

np.random.seed(6)
lbl_cmap = random_label_cmap()


## Paths and channel settings


In [ ]:
# Fig. 2/Fig. 3 input images and ring-canal segmentation outputs.
data_directory = Path("../Images/RC_training/test")
results_directory = data_directory / "Results"
expt_name = "Pnut_test"

# Channel indices in the source TIFF stacks. The original notebook used Fas3=3 and Vsg=1.
FAS3_CHANNEL = 3
VSG_CHANNEL = 1

# Local copy of the StarDist ring-canal model used by the original notebook.
model_basedir = Path("../Models/Stardist")
model_name = "ring_canals"

results_directory.mkdir(parents=True, exist_ok=True)
img_files = sorted(data_directory.glob("*.tif"))
img_list = [imread(path) for path in img_files]

print(f"Loaded {len(img_list)} image stacks from {data_directory}")


## Background correction


In [ ]:
# Fig. 2/Fig. 3 preprocessing: subtract Vsg-channel background before prediction.
for i, img in enumerate(img_list):
    otsu = filters.threshold_otsu(img[..., VSG_CHANNEL])
    corrected = img.copy()
    corrected[..., VSG_CHANNEL] = subtract_background(corrected[..., VSG_CHANNEL], otsu)
    img_list[i] = corrected


## Load StarDist model and predict labels


In [ ]:
# Fig. 2/Fig. 3 segmentation model for Fas3/Vsg ring-canal images.
model = StarDist3D(None, name=model_name, basedir=str(model_basedir))

axis_norm = (0, 1, 2)
labels_array = []
details_array = []

for img_raw in img_list:
    gc.collect(2)
    model_input = img_raw[..., [FAS3_CHANNEL, VSG_CHANNEL]]
    model_input = normalize(model_input, 1, 99.8, axis=axis_norm)
    labels, details = model.predict_instances(model_input)
    labels_array.append(labels)
    details_array.append(details)

print(f"Predicted labels for {len(labels_array)} image stacks")


## Optional manual split corrections

Populate `manual_splits` only after inspecting the labels. Keys are zero-based image indices; values are label IDs that should be split by erosion/dilation.


In [ ]:
# Fig. 2/Fig. 3 manual cleanup: leave empty for a fully automated rerun.
manual_splits = {
    # Example from the source notebook:
    # 3: {"labels": [22], "erosion_radius": 15},
}

for image_index, split_args in manual_splits.items():
    labels_array[image_index] = SplitLabels(
        labels_array[image_index],
        split_args["labels"],
        erosion_radius=split_args.get("erosion_radius", 1),
    )


## Quality-control preview


In [ ]:
# Fig. 2/Fig. 3 QC: inspect center z-slices before saving ImageJ-compatible stacks.
n_preview = min(3, len(labels_array))
fig, axes = plt.subplots(n_preview, 2, figsize=(10, 3.5 * n_preview), squeeze=False)

for row in range(n_preview):
    z = labels_array[row].shape[0] // 2
    axes[row, 0].imshow(labels_array[row][z], cmap=lbl_cmap)
    axes[row, 1].imshow(img_list[row][z, :, :, VSG_CHANNEL], cmap="gray")
    axes[row, 0].set_title("Predicted labels")
    axes[row, 1].set_title("Vsg channel")
    axes[row, 0].axis("off")
    axes[row, 1].axis("off")

plt.tight_layout()


## Save combined image + label stacks


In [ ]:
# Fig. 2/Fig. 3 output: append labels as a final channel for downstream review/scoring.
for i, labels in enumerate(labels_array, start=1):
    label_channel = labels[..., np.newaxis] if labels.ndim == 3 else labels
    combined = np.concatenate((img_list[i - 1], label_channel), axis=3)
    save_path = results_directory / f"{expt_name}_{i}.tif"
    save_tiff_imagej_compatible(save_path, combined, axes="ZYXC")
    print(f"Saved {save_path}")
